# Multi-Head Attention

## 왜 필요한가?

- 하나의 Head는 한 가지 관계만 학습하기 쉽다.
- 여러 Head가 서로 다른 관계를 병렬로 학습한다.
- Concat 후 Linear Layer로 정보를 통합한다.

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

In [3]:
import torch
from torch import nn
from src.attention import scaled_dot_product_attention

attention.py loaded


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()
        
        # d_model: 토큰 하나를 표현하는 벡터의 차원
        self.d_model = d_model

        # Multi-Head Attention에서 사용할 Head의 개수
        self.num_heads = num_heads

        # Head 하나가 담당하는 벡터 차원 (512 // 8 = 64)
        self.head_dim = d_model // num_heads

        # d_model이 Head 개수로 나누어지는지 확인
        assert d_model % num_heads == 0

        # Query, Key, Value를 생성하기 위한 Linear Layer
        # 논문의 QW^Q, KW^K, VW^V에 해당
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 여러 Head의 결과를 다시 하나의 벡터로 변환하는 Linear Layer
        # 논문의 W^O에 해당
        self.W_o = nn.Linear(d_model, d_model)

   
    def forward(self, Q, K, V):
        # Qeury, Key, Value 생성
        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)

        # 입력 Tensor의 크기 저장
        batch_size = Q.size(0)
        seq_len = Q.size(1)

        # (batch_size, seq_len, d_model)
        # -> (batch_size, seq_len, num_heads, head_dim)
        # 여러 Head가 각각 독립적으로 Attention을 계산할 수 있도록 분리
        Q = Q.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        K = K.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        V = V.reshape(batch_size, seq_len, self.num_heads, self.head_dim)

        # (batch_size, seq_len, num_haeds, head_dim)
        # -> (batch_size, num_heads, seq_len, head_dim)
        # 각 Head가 (seq_len, head_dim) 형태의 입력을 받을 수 있도록 차원 변경
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        
        # 각 Head에서 Scaled Dot-Product Attention 수행
        scores, attn_weights, output = scaled_dot_product_attention(Q, K, V)

        # (batch_size, num_heads, seq_len, head_dim)
        # -> (batch_size, seq_len, num_heads, head_dim)
        # Head 결과를 다시 합치기 위해 차원복원
        output = output.transpose(1, 2)

        # 여러 Head를 하나의 벡터(d_model)로 이어 붙임(concat)
        # (batch_size, seq_len, num_heads, head_dim)
        # -> (batch_size, seq_len, d_model)
        output = output.reshape(batch_size, seq_len, self.num_heads * self.head_dim)

        # 최종 Linear Projection (논문의 W^O)
        output = self.W_o(output)

        return output

Q = torch.randn(32, 10, 512)
K = torch.randn(32, 10, 512)
V = torch.randn(32, 10, 512)

model = MultiHeadAttention()
output = model(Q, K, V)
print(output.shape)


torch.Size([32, 10, 512])
